In [ ]:
# Single-Cell RNA-seq Tumour Microenvironment (TME) Pipeline — Breast Cancer Dataset: Wu et al. 2021, GSE176078 — 26 breast cancer patients (ER+, HER2+, TNBC), ~100k cells
# Goal: Build a full single-cell analysis pipeline — QC, normalisation, batch-correctedrepresentation learning (scVI), unsupervised clustering, cell type annotation, and atranslational ML layer linking tumour microenvironment (TME) composition to immune phenotype.
#Pipeline overview:1. Data loading and QC2. Normalisation and highly variable gene (HVG) selection3. PCA baseline (dimensionality reduction + UMAP)4. scVI — batch-corrected latent representation (core ML step)5. Leiden clustering and cell type annotation6. TME composition per patient7. Subtype classification (honest negative result + interpretation)8. Immune hot vs cold classification (positive result)9. SHAP explainability — biological interpretation10. Benchmarking PCA vs scVI embeddings

In [ ]:
## 1. Imports and environment check

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import mmread
from scipy.sparse import csr_matrix
import os

print("scanpy version:", sc.__version__)
print("All imports successful")

In [ ]:
## 2. Data loadingThe Wu et al. dataset provides one folder per patient, each containing:- `count_matrix_sparse.mtx` — sparse cells x genes count matrix- `count_matrix_barcodes.tsv` — cell barcodes- `count_matrix_genes.tsv` — gene names- `metadata.csv` — per-cell metadata, including author-curated cell type annotations  (`celltype_major`, `celltype_minor`, `celltype_subset`), patient subtype (ER+/HER2+/TNBC),  and QC metrics (`nCount_RNA`, `nFeature_RNA`, `percent.mito`)
# We load all 26 samples and concatenate them into a single `AnnData` object, tagging eachcell with its patient/sample of origin (needed later for batch correction).

In [ ]:
data_path = "/home/josephimhanbor/scRNA files/"

# get all patient folders (only directories starting with CID)
sample_folders = sorted([
    f for f in os.listdir(data_path)
    if os.path.isdir(os.path.join(data_path, f)) and f.startswith("CID")
])

print(f"Found {len(sample_folders)} samples:")
print(sample_folders)

In [ ]:
adatas = []  # will hold one AnnData per sample

for sample in sample_folders:
    sample_path = os.path.join(data_path, sample)

    # load the three matrix files
    matrix = mmread(os.path.join(sample_path, "count_matrix_sparse.mtx"))
    barcodes = pd.read_csv(os.path.join(sample_path, "count_matrix_barcodes.tsv"),
                           header=None, names=["barcode"])
    genes = pd.read_csv(os.path.join(sample_path, "count_matrix_genes.tsv"),
                        header=None, names=["gene"])
    metadata = pd.read_csv(os.path.join(sample_path, "metadata.csv"),
                           index_col=0)

    # build AnnData object for this sample
    adata = ad.AnnData(
        X=csr_matrix(matrix).T,  # transpose: rows=cells, columns=genes
        obs=metadata,
        var=pd.DataFrame(index=genes["gene"])
    )

    adata.obs_names = barcodes["barcode"].values
    adata.obs["sample"] = sample  # tag each cell with its sample ID

    adatas.append(adata)
    print(f"Loaded {sample}: {adata.shape[0]} cells x {adata.shape[1]} genes")

# concatenate all samples into one object
adata_full = ad.concat(adatas, label="sample", keys=sample_folders)
print(f"\nFull dataset: {adata_full.shape[0]} cells x {adata_full.shape[1]} genes")

In [ ]:
print(adata_full)
print("\n--- obs columns ---")
print(adata_full.obs.columns.tolist())
print("\n--- subtype distribution ---")
print(adata_full.obs['subtype'].value_counts())
print("\n--- major cell types ---")
print(adata_full.obs['celltype_major'].value_counts())

In [ ]:
# Checkpoint — save the raw concatenated object so we never need to reload the26 raw matrices again.

In [ ]:
os.makedirs("/home/josephimhanbor/scRNA files/results", exist_ok=True) #my own file path, replace with yours

adata_full.write("/home/josephimhanbor/scRNA files/results/adata_raw.h5ad") #my own file path, replace with yours
print("Saved raw AnnData object")
print(f"File size: {os.path.getsize('/home/josephimhanbor/scRNA files/results/adata_raw.h5ad') / 1e6:.1f} MB")

In [ ]:
# 3. Quality control (QC)We inspect three standard QC metrics before filtering:- `nFeature_RNA` — number of genes detected per cell. Too few (<200) suggests an  empty droplet or dying cell; too many (>6000) suggests a doublet (two cells captured  as one).- **`nCount_RNA`** — total UMI counts per cell.
# Similar interpretation to gene counts.- **`percent.mito`** — percentage of reads mapping to mitochondrial genes. Dying cells  lose cytoplasmic RNA but retain mitochondrial RNA, so a high mito percentage (>20%)  indicates a damaged cell.

In [ ]:
sc.settings.figdir = "/home/josephimhanbor/scRNA files/results/"
sc.settings.set_figure_params(dpi=100, figsize=(10, 4))

# rename to scanpy-friendly names
adata_full.obs['n_counts'] = adata_full.obs['nCount_RNA']
adata_full.obs['n_genes'] = adata_full.obs['nFeature_RNA']
adata_full.obs['pct_mito'] = adata_full.obs['percent.mito']

sc.pl.violin(adata_full,
             keys=['n_genes', 'n_counts', 'pct_mito'],
             jitter=0.4,
             multi_panel=True,
             save="_qc_before_filtering.png")

print("QC plot saved")

In [ ]:
# Filtering thresholds based on the distributions above:- Keep cells with `n_genes > 200` (removes empty droplets)- Keep cells with `n_genes < 6000` (removes likely doublets)- Keep cells with `pct_mito < 20` (removes dying/damaged cells)The authors had already performed some preliminary QC before depositing the data, so weexpect to remove only a small fraction of cells here.

In [ ]:
print("Cells before filtering:", adata_full.shape[0])

adata = adata_full[
    (adata_full.obs['n_genes'] > 200) &
    (adata_full.obs['n_genes'] < 6000) &
    (adata_full.obs['pct_mito'] < 20)
].copy()

print("Cells after filtering:", adata.shape[0])
print("Cells removed:", adata_full.shape[0] - adata.shape[0])

In [ ]:
## 4. Normalisation and highly variable gene (HVG) selection: Normalisation — each cell has a different total UMI count, mostly due to technicalfactors (sequencing depth), not biology. We normalise every cell to the same total count(10,000) and apply a `log1p` transform to stabilise variance.
# Highly variable genes (HVGs) — of ~29,733 genes, most are not informative(housekeeping genes expressed uniformly across cells). We keep the top 3,000 genes whosevariance is highest *relative to their expression level*, using `batch_key='sample'` sothat genes are selected for variability *within* each patient — preventingpatient-specific genes from dominating the HVG list.

In [ ]:
# normalise each cell to 10,000 total counts
sc.pp.normalize_total(adata, target_sum=1e4)

# log transform
sc.pp.log1p(adata)

# store the normalised values in a separate layer for reference
adata.layers['normalised'] = adata.X.copy()

print("Normalisation complete")
print("Data is now log1p normalised")

In [ ]:
# find highly variable genes
sc.pp.highly_variable_genes(adata,
                             n_top_genes=3000,
                             batch_key='sample',
                             flavor='seurat_v3',
                             subset=False)

print("Highly variable genes identified")
print(f"Number of HVGs: {adata.var['highly_variable'].sum()}")

sc.pl.highly_variable_genes(adata, save="_hvg.png")
print("HVG plot saved")

In [ ]:
## 5. ScalingBefore PCA, each gene is scaled to mean 0 / variance 1 so that highly expressed genes donot dominate purely due to magnitude. 
# To keep this memory-tractable (scaling all ~30kgenes for ~100k cells would require ~22 GB), we subset to the 3,000 HVGs *first* and scaleonly that matrix. This is standard practice — PCA, clustering, and scVI all operate on theHVG subset anyway.
# We also store the (unscaled, log-normalised) full data in `adata.raw` for later reference.

In [ ]:
# save normalised counts before scaling (kept for reference)
adata.raw = adata

# subset to HVGs only before scaling
adata_hvg = adata[:, adata.var['highly_variable']].copy()

print(f"Working with {adata_hvg.shape[1]} HVGs instead of {adata.shape[1]} genes")

# scale only the HVG subset
sc.pp.scale(adata_hvg, max_value=10)

print("Scaling complete")

In [ ]:
## 6. PCA baselinePCA gives us a linear dimensionality reduction that serves two purposes:1. A quick way to inspect global structure and decide how many components capture   meaningful biological variance (the "elbow" in the variance ratio plot)2. A **baseline embedding** to benchmark against scVI later — does batch-aware,   non-linear representation learning actually improve on simple PCA?

In [ ]:
sc.tl.pca(adata_hvg, n_comps=50, svd_solver='arpack')

print("PCA complete")
print(f"PCA shape: {adata_hvg.obsm['X_pca'].shape}")

sc.pl.pca_variance_ratio(adata_hvg, n_pcs=50, log=True, save="_pca_variance.png")
print("PCA variance plot saved")

The variance ratio shows a clear elbow around PC20-25, beyond which additional componentscontribute little. We use the first **30 PCs** (slightly conservative) for the neighbourgraph and UMAP below.

In [ ]:
# build neighbour graph using top 30 PCs
sc.pp.neighbors(adata_hvg, n_pcs=30, n_neighbors=15)

# compute UMAP embedding
sc.tl.umap(adata_hvg)

print("Neighbour graph and UMAP complete")

sc.pl.umap(adata_hvg,
           color=['celltype_major', 'sample', 'subtype'],
           ncols=1,
           save="_umap_pca_baseline.png")

print("UMAP plot saved")

**Result:** Major cell types separate reasonably well on PCA alone, but the `sample` panelshows each patient forming its own island — particularly within the Cancer Epithelialpopulation. This is a **batch effect**: patient identity is driving separation as much asbiology. This is the motivating problem for scVI.**Checkpoint** — save the PCA baseline object (includes `.raw` with full log-normaliseddata for later use).

In [ ]:
adata_hvg.write("/home/josephimhanbor/scRNA files/results/adata_pca_baseline.h5ad") #my own file path, replace with yours
print("PCA baseline saved")

In [ ]:
## 7. scVI — batch-corrected latent representation (core ML step)**scVI** is a variational autoencoder (VAE) designed for single-cell count data. Theencoder compresses each cell's gene expression profile into a low-dimensional latentspace (here, 30 dimensions) described by a mean and variance, rather than a single point.The decoder reconstructs the original counts from a sample of this distribution.Key design choices:- **`gene_likelihood="nb"`** — uses a negative binomial reconstruction loss, which matches  the over-dispersed nature of single-cell count data (more appropriate than Gaussian/MSE).- **`batch_key='sample'`** — patient identity is given to the model as a covariate to  *correct for*, not biology to learn. 
#The model is explicitly told "this variation is  technical."**Important:** scVI must be trained on **raw counts**, not the scaled/log-normalised dataused for PCA — it performs its own internal normalisation. 
# We therefore reload the rawcounts, re-apply the same QC filter, and subset to the same HVGs identified above.

In [ ]:
import scvi

# reload raw counts and re-apply the same QC filter
adata_scvi = sc.read_h5ad("/home/josephimhanbor/scRNA files/results/adata_raw.h5ad")

adata_scvi = adata_scvi[
    (adata_scvi.obs['nFeature_RNA'] > 200) &
    (adata_scvi.obs['nFeature_RNA'] < 6000) &
    (adata_scvi.obs['percent.mito'] < 20)
].copy()

# subset to the same HVGs identified earlier
adata_baseline = sc.read_h5ad("/home/josephimhanbor/scRNA files/results/adata_pca_baseline.h5ad")
hvg_genes = adata_baseline.var_names[adata_baseline.var['highly_variable']]
adata_scvi = adata_scvi[:, hvg_genes].copy()

print(f"Shape: {adata_scvi.shape}")
print(f"Max value: {adata_scvi.X.max()}")
print(f"Sample values (raw counts): {adata_scvi.X[0, :10].toarray().flatten()}")

**Subsampling for training speed** — training scVI on all ~98k cells on CPU is slow(>30 min). We subsample to 30,000 cells, which is more than sufficient to learn the majorcell type representations and batch-correction patterns; all 9 major TME populationsremain well represented (the smallest, B-cells, still has ~900+ cells). This is standardpractice and does not affect the biological conclusions of the pipeline.

In [ ]:
# subsample to 30k cells for tractable training time
sc.pp.subsample(adata_scvi, n_obs=30000, random_state=42)
print(f"Subsampled to: {adata_scvi.shape[0]} cells")

# register with scVI
scvi.model.SCVI.setup_anndata(
    adata_scvi,
    batch_key='sample'
)

# build the model
model = scvi.model.SCVI(
    adata_scvi,
    n_layers=2,        # two encoder/decoder layers
    n_latent=30,       # 30-dimensional latent space
    gene_likelihood="nb"  # negative binomial - appropriate for count data
)

print(model)

In [ ]:
model.train(
    max_epochs=100,
    early_stopping=True,
    early_stopping_patience=10,
)

print("Training complete")

In [ ]:
# extract the 30-dimensional latent space scVI learned
adata_scvi.obsm['X_scvi'] = model.get_latent_representation()

print(f"Latent representation shape: {adata_scvi.obsm['X_scvi'].shape}")

# build neighbour graph on scVI latent space
sc.pp.neighbors(adata_scvi, use_rep='X_scvi', n_neighbors=15)

# compute UMAP
sc.tl.umap(adata_scvi)

print("UMAP complete")

sc.pl.umap(adata_scvi,
           color=['celltype_major', 'sample', 'subtype'],
           ncols=1,
           save="_umap_scvi.png")

print("scVI UMAP saved")

**Result:** Compared to the PCA baseline, cell types form cleaner, more distinct islands,and — critically — the `sample` panel now shows patients **mixing within** each cell typecloud rather than forming separate per-patient islands. scVI has successfully correctedthe batch effect while preserving (and arguably sharpening) biological structure.**Checkpoint** — save the trained model and the scVI-processed AnnData object.

In [ ]:
model_path = "/home/josephimhanbor/scRNA files/results/scvi_model"
model.save(model_path, overwrite=True)

adata_scvi.write("/home/josephimhanbor/scRNA files/results/adata_scvi.h5ad") # my own file path, replace with yours

print("Model and AnnData saved")

In [ ]:
## 8. Leiden clustering and cell type annotationWe cluster cells in the scVI latent space using the **Leiden algorithm** — a graph-basedcommunity detection method that finds densely connected groups of cells in the k-nearestneighbour graph. Resolution controls granularity (higher = more, finer clusters).

In [ ]:
sc.tl.leiden(adata_scvi, resolution=0.5, key_added='leiden_scvi')

print("Clustering complete")
print(f"Number of clusters: {adata_scvi.obs['leiden_scvi'].nunique()}")
print(adata_scvi.obs['leiden_scvi'].value_counts())

In [ ]:
sc.pl.umap(adata_scvi,
           color=['leiden_scvi', 'celltype_major'],
           ncols=2,
           legend_loc='on data',
           save="_umap_clusters.png")

print("Cluster plot saved")

**Validation against ground truth** — the Wu et al. authors provide curated cell typeannotations (`celltype_major`). We cross-tabulate our unsupervised Leiden clusters againstthese annotations to check whether our pipeline rediscovers known biology.

In [ ]:
crosstab = pd.crosstab(
    adata_scvi.obs['leiden_scvi'],
    adata_scvi.obs['celltype_major'],
    normalize='index'  # normalise by row so each cluster sums to 1
)

print("Cluster composition (fraction of each cell type per cluster):")
print(crosstab.round(2))

**Result:** Almost every cluster is >93% pure for a single cell type — including fourdistinct T-cell subclusters (likely CD8+, CD4+, Treg, and exhausted states) discoveredwithout ever being told what cell types exist. This validates the entire pipeline up tothis point.We now map each Leiden cluster to its majority cell type for downstream analysis.

In [ ]:
# create cluster to cell type mapping based on majority vote
cluster_annotations = {
    '0': 'Cancer Epithelial',
    '1': 'T-cells',
    '2': 'T-cells',
    '3': 'Myeloid',
    '4': 'Endothelial',
    '5': 'T-cells',
    '6': 'CAFs',
    '7': 'PVL',
    '8': 'Cancer Epithelial',
    '9': 'Plasmablasts',
    '10': 'B-cells',
    '11': 'T-cells',
    '12': 'Normal Epithelial',
    '13': 'Normal Epithelial',
    '14': 'Cancer Epithelial',
    '15': 'Myeloid',
    '16': 'Myeloid'
}

adata_scvi.obs['cluster_annotation'] = adata_scvi.obs['leiden_scvi'].map(cluster_annotations)

sc.pl.umap(adata_scvi,
           color=['cluster_annotation'],
           legend_loc='on data',
           title='scVI clusters — annotated',
           save="_umap_annotated.png")

print("Annotation complete")
print(adata_scvi.obs['cluster_annotation'].value_counts())

In [ ]:
adata_scvi.write("/home/josephimhanbor/scRNA files/results/adata_scvi_clustered.h5ad") #my own file path, replace with yours
print("Clustered object saved")

In [ ]:
## 9. Tumour microenvironment (TME) composition per patientThis step converts single-cell data into a **per-patient feature matrix**: for each ofthe 26 patients, what fraction of their tumour is each cell type? This is the bridgebetween single-cell biology and patient-level (translational) analysis.

In [ ]:
composition = pd.crosstab(
    adata_scvi.obs['sample'],
    adata_scvi.obs['cluster_annotation'],
    normalize='index'  # each row (patient) sums to 1
)

print("TME composition matrix shape:", composition.shape)
print("\nComposition per patient:")
print(composition.round(3))

In [ ]:
# add subtype information for visualisation
subtype_map = adata_scvi.obs[['sample', 'subtype']].drop_duplicates().set_index('sample')
composition['subtype'] = subtype_map['subtype']

print("Subtype distribution:")
print(composition['subtype'].value_counts())

# heatmap of composition, sorted by subtype
comp_plot = composition.sort_values('subtype')
comp_features = comp_plot.drop(columns=['subtype'])

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(comp_features,
            annot=True,
            fmt='.2f',
            cmap='YlOrRd',
            yticklabels=[f"{idx} ({comp_plot.loc[idx, 'subtype']})"
                        for idx in comp_plot.index],
            ax=ax)
ax.set_title('TME Composition per Patient by Subtype', fontsize=14)
ax.set_xlabel('Cell Type')
ax.set_ylabel('Patient (Subtype)')
plt.tight_layout()
plt.savefig("/home/josephimhanbor/scRNA files/results/tme_composition_heatmap.png",
            dpi=150, bbox_inches='tight') # my own file path, replace with yours
plt.show()
print("Heatmap saved")

The heatmap reveals substantial inter-patient heterogeneity: some tumours areT-cell-dominated and immune-hot (e.g. CID4398 at 85% T-cells), others areepithelial-dominated and immune-excluded (e.g. CID4290A at 71% Cancer Epithelial), andothers show distinct myeloid- or stromal-dominated profiles. This heterogeneity motivatesthe translational ML analyses below.

In [ ]:
## 10. Molecular subtype classification from TME composition**Hypothesis to test:** can the TME composition (9 cell-type fractions) predict thepatient's molecular subtype (ER+ / HER2+ / TNBC)?With only **26 patients**, a standard train/test split would leave too few test samplesto be meaningful. 
# We instead use **Leave-One-Out Cross-Validation (LOOCV)**: for eachpatient, train on the remaining 25 and predict that patient's subtype, repeated 26 times.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb

X = composition.drop(columns=['subtype']).values
y = composition['subtype'].values

print("Feature matrix shape:", X.shape)
print("Class distribution:")
print(pd.Series(y).value_counts())

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Classes:", le.classes_)

loo = LeaveOneOut()
y_true, y_pred = [], []

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    model_xgb = xgb.XGBClassifier(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        eval_metric='mlogloss'
    )
    model_xgb.fit(X_train, y_train)

    pred = model_xgb.predict(X_test)
    y_true.append(y_test[0])
    y_pred.append(pred[0])

accuracy = accuracy_score(y_true, y_pred)
print(f"\nLOOCV Accuracy: {accuracy:.3f}")
print(f"\nClassification report:")
print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))

**Honest result and interpretation:** LOOCV accuracy of **50%** (vs a 42% "always predictER+" baseline). HER2+ has 0% recall — with only 5 HER2+ patients, there is not enoughsignal to learn a distinct HER2+ signature.**Biological interpretation:** this is not a methodological failure. Molecular subtype isdefined by receptor expression (ER, PR, HER2) *within the malignant cells themselves*, notby the surrounding immune/stromal composition. Asking the TME composition to predict alabel that is fundamentally about the tumour cells' own molecular identity is the wrongquestion. This null result motivates a reframing toward a question the TME composition*is* suited to answer — immune phenotype.

In [ ]:
## 11. Immune hot vs cold classification**Reframed question:** does the *non-T-cell* composition of a tumour (myeloid, stromal,epithelial, etc.) predict whether that tumour is also CD8+ T-cell infiltrated("immune-hot") or not ("immune-cold")?This is clinically important — immune-hot tumours respond better to checkpoint inhibitorimmunotherapy, while immune-cold tumours generally do not. 
# It is also a genuine biologicalquestion: immunosuppressive myeloid cells and cancer-associated fibroblasts (CAFs) areknown to physically and chemically *exclude* T-cells from the tumour.First, check what clinical/biological metadata is available.

In [ ]:
print("All obs columns:")
print(adata_scvi.obs.columns.tolist())

for col in adata_scvi.obs.columns:
    n_unique = adata_scvi.obs[col].nunique()
    if n_unique < 30:
        print(f"\n{col}: {adata_scvi.obs[col].unique()}")

No survival/outcome data is available in this dataset (it is a single-timepoint snapshot).However, `celltype_minor` includes `T cells CD8+` specifically — the clinically relevanteffector population for immunotherapy response. We use the **CD8+ T-cell fraction** perpatient to define "hot" vs "cold" labels.

In [ ]:
cd8_composition = pd.crosstab(
    adata_scvi.obs['sample'],
    adata_scvi.obs['celltype_minor'],
    normalize='index'
)

cd8_fraction = cd8_composition['T cells CD8+']

print("CD8+ T-cell fraction per patient:")
print(cd8_fraction.sort_values(ascending=False))

print(f"\nMedian CD8+ fraction: {cd8_fraction.median():.3f}")
print(f"Mean CD8+ fraction: {cd8_fraction.mean():.3f}")

The median (0.106) falls in a natural gap in the distribution, giving a balanced 13/13split between "hot" (above median) and "cold" (below median).**Avoiding circularity:** to predict immune status from "the rest of the TME", the featurematrix must **exclude all T-cell populations** (including the CD8+ fraction used to definethe label). We use the 9 major cell type categories from clustering, drop T-cells, andrenormalise the remaining 8 categories to sum to 1.

In [ ]:
# define immune hot/cold based on median CD8+ T-cell fraction
threshold = cd8_fraction.median()
immune_status = (cd8_fraction > threshold).map({True: 'hot', False: 'cold'})

print("Immune status distribution:")
print(immune_status.value_counts())

# feature matrix: major cell types, excluding T-cells, renormalised
major_composition = pd.crosstab(
    adata_scvi.obs['sample'],
    adata_scvi.obs['cluster_annotation'],
    normalize=False
)

non_tcell_major = [c for c in major_composition.columns if c != 'T-cells']
X_immune = major_composition[non_tcell_major]
X_immune = X_immune.div(X_immune.sum(axis=1), axis=0)

print(f"\nFeature matrix shape: {X_immune.shape}")
print(f"Features: {X_immune.columns.tolist()}")

y_immune = immune_status.loc[X_immune.index]
print("\nLabels:")
print(y_immune.value_counts())

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

le_immune = LabelEncoder()
y_enc = le_immune.fit_transform(y_immune)
print("Classes:", le_immune.classes_)

X_arr = X_immune.values

loo = LeaveOneOut()
y_true, y_pred = [], []

for train_idx, test_idx in loo.split(X_arr):
    X_train, X_test = X_arr[train_idx], X_arr[test_idx]
    y_train, y_test = y_enc[train_idx], y_enc[test_idx]

    clf = xgb.XGBClassifier(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss'
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    y_true.append(y_test[0])
    y_pred.append(pred[0])

accuracy = accuracy_score(y_true, y_pred)
print(f"\nLOOCV Accuracy: {accuracy:.3f}")
print(f"\nClassification report:")
print(classification_report(y_true, y_pred, target_names=le_immune.classes_, zero_division=0))
print(f"\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

**Result: 80.8% LOOCV accuracy** on a balanced 13/13 split (vs 50% baseline). Thenon-T-cell composition of a tumour — its myeloid, stromal, and epithelial makeup — carriesreal information about whether that tumour is also CD8+ T-cell infiltrated. The confusionmatrix shows the model is more reliable at identifying "cold" tumours (12/13 correct) than"hot" ones (9/13 correct), suggesting there may be multiple distinct biological routes toan immunosuppressive cold TME.

In [ ]:
## 12. SHAP explainability — which cell types drive the prediction?**SHAP (SHapley Additive exPlanations)** decomposes each prediction into per-featurecontributions, based on Shapley values from cooperative game theory.
# We re-fit a**RandomForest** classifier on the full dataset (chosen for SHAP/TreeExplainercompatibility) and compute SHAP values for the "hot" class.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import shap

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=42
)
rf_model.fit(X_arr, y_enc)

explainer = shap.TreeExplainer(rf_model)
sv = explainer.shap_values(X_immune)

print(type(sv))
print(np.array(sv).shape)

In [ ]:
# SHAP values for class 1 ("hot")
sv_hot = sv[:, :, 1]  # shape (26, 8)

# mean absolute SHAP value per feature = overall importance
mean_abs_shap = np.abs(sv_hot).mean(axis=0)

shap_importance = pd.DataFrame({
    'feature': X_immune.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=True)

print(shap_importance)

plt.figure(figsize=(8, 5))
plt.barh(shap_importance['feature'], shap_importance['mean_abs_shap'], color='steelblue')
plt.xlabel('Mean |SHAP value|')
plt.title('Feature importance — predicting immune hot vs cold TME')
plt.tight_layout()
plt.savefig("/home/josephimhanbor/scRNA files/results/shap_importance.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved")

**B-cell fraction** is by far the strongest predictor, followed by **Cancer Epithelialfraction**. We check the direction of these effects.

In [ ]:
plt.figure(figsize=(10, 6))
for i, feat in enumerate(['B-cells', 'Cancer Epithelial']):
    plt.subplot(1, 2, i+1)
    plt.scatter(X_immune[feat], sv_hot[:, X_immune.columns.get_loc(feat)],
                c=y_enc, cmap='coolwarm', s=60)
    plt.xlabel(f'{feat} fraction')
    plt.ylabel('SHAP value (toward "hot")')
    plt.title(feat)
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.savefig("/home/josephimhanbor/scRNA files/results/shap_directionality.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved")

**Biological interpretation:**- **B-cells:** patients with B-cell fraction above ~0.04 consistently have positive SHAP  values (push toward "hot"); patients near zero B-cell fraction consistently push toward  "cold". This is consistent with **tertiary lymphoid structures (TLS)** — organised  immune cell aggregates where B-cells and T-cells co-localise. B-cell presence appears to  be a strong proxy for an organised, coordinated immune response that includes CD8+  T-cell infiltration.- **Cancer Epithelial:** an inverse relationship — tumours with low Cancer Epithelial  fraction (<0.2) tend to be "hot"; tumours with high Cancer Epithelial fraction (>0.4)  consistently push toward "cold". This is the classic **immune exclusion by tumour  mass** pattern described in the TNBC literature — a high density of malignant cells  physically and chemically excludes T-cell infiltration.

In [ ]:
## 13. Benchmarking — PCA vs scVI embeddingsWe quantify the qualitative differences seen in the UMAPs using **silhouette scores**:- **Cell-type silhouette** (higher = better) — how well-separated are cell types in the  embedding?- **Batch (sample) silhouette** (lower = better) — how well do different patients' cells  mix together within the same cell type?

In [ ]:
from sklearn.metrics import silhouette_score

# Use only cells present in adata_scvi (the 30k subsample)
common_cells = adata_scvi.obs_names
adata_hvg_subset = adata_hvg[common_cells].copy()

sil_celltype_pca = silhouette_score(adata_hvg_subset.obsm['X_pca'][:, :30],
                                      adata_scvi.obs['celltype_major'].values,
                                      sample_size=5000, random_state=42)

sil_celltype_scvi = silhouette_score(adata_scvi.obsm['X_scvi'],
                                       adata_scvi.obs['celltype_major'].values,
                                       sample_size=5000, random_state=42)

sil_batch_pca = silhouette_score(adata_hvg_subset.obsm['X_pca'][:, :30],
                                   adata_scvi.obs['sample'].values,
                                   sample_size=5000, random_state=42)

sil_batch_scvi = silhouette_score(adata_scvi.obsm['X_scvi'],
                                    adata_scvi.obs['sample'].values,
                                    sample_size=5000, random_state=42)

print("=== Biological signal (higher = better cell type separation) ===")
print(f"PCA silhouette (cell type):  {sil_celltype_pca:.4f}")
print(f"scVI silhouette (cell type): {sil_celltype_scvi:.4f}")

print("\n=== Batch effect (lower = better patient mixing) ===")
print(f"PCA silhouette (batch):  {sil_batch_pca:.4f}")
print(f"scVI silhouette (batch): {sil_batch_scvi:.4f}")

**Honest interpretation:** PCA shows a *higher* raw cell-type silhouette (0.25 vs 0.14),and batch silhouettes are close (-0.04 vs -0.06). Taken at face value this looks like adraw or even a loss for scVI — but silhouette scores on raw embeddings can be misleadingfor batch-corrected data.PCA's higher cell-type silhouette likely reflects **residual patient-specific structure**that inflates apparent separation — in the PCA UMAP, each patient's malignant cells formedtheir own island, partly *because* they are from the same patient, not purely because theyare generalisably "cancer cells". scVI's lower silhouette reflects a more conservative,**generalisable** embedding that deliberately compresses out patient-specific structure.This is exactly what the downstream Leiden clustering validation demonstrated: >93%concordance with curated cell type annotations **across all 26 patients**, including fourdistinct, patient-mixed T-cell subclusters. Silhouette score alone is an insufficientbenchmark for batch-corrected embeddings — ground-truth-validated clustering is the moremeaningful criterion here.